<a href="https://colab.research.google.com/github/KevanPatira/BIS-Lab/blob/main/Genetic_Algorithm_for_optimization_distance_between_src_and_dest_in_a_city_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import random

# =====================================================
# 1. CITY ROAD NETWORK
# Nodes = intersections / locations
# Each road has distance (km) and average speed (km/h)
# Roads are bidirectional in this example.
# =====================================================

graph = {}

def add_road(u, v, distance, speed):
    graph.setdefault(u, {})[v] = {
        "distance": distance,
        "speed": speed
    }
    graph.setdefault(v, {})[u] = {
        "distance": distance,
        "speed": speed
    }


# Sample city network: replace with actual road data
add_road("A", "B", 2.0, 30)
add_road("A", "C", 3.0, 40)
add_road("A", "D", 4.0, 50)
add_road("B", "C", 1.0, 20)
add_road("B", "E", 3.0, 40)
add_road("C", "D", 2.0, 30)
add_road("C", "E", 2.0, 35)
add_road("C", "F", 4.0, 50)
add_road("D", "F", 3.0, 40)
add_road("D", "G", 5.0, 50)
add_road("E", "F", 2.0, 30)
add_road("E", "H", 5.0, 45)
add_road("F", "G", 2.0, 35)
add_road("F", "H", 3.0, 40)
add_road("G", "H", 2.0, 25)
add_road("G", "I", 4.0, 50)
add_road("H", "I", 2.0, 30)
add_road("H", "J", 4.0, 45)
add_road("I", "J", 2.0, 35)


# =====================================================
# 2. GENETIC ALGORITHM PARAMETERS
# =====================================================

POPULATION_SIZE = 100
GENERATIONS = 300
MUTATION_RATE = 0.25
TOURNAMENT_SIZE = 4
ELITE_COUNT = 2


# =====================================================
# 3. ROUTE DISTANCE AND JOURNEY TIME
# =====================================================

def route_distance(route):
    distance = 0.0

    for i in range(len(route) - 1):
        road = graph[route[i]].get(route[i + 1])

        if road is None:
            return float("inf")

        distance += road["distance"]

    return distance


def route_time(route):
    # Return estimated driving time in minutes
    total_time = 0.0

    for i in range(len(route) - 1):
        road = graph[route[i]].get(route[i + 1])

        if road is None:
            return float("inf")

        total_time += (
            road["distance"] / road["speed"]
        ) * 60

    return total_time


def valid_route(route, source, destination):
    if not route:
        return False

    if route[0] != source or route[-1] != destination:
        return False

    if len(route) != len(set(route)):
        return False

    return all(
        route[i + 1] in graph.get(route[i], {})
        for i in range(len(route) - 1)
    )


# =====================================================
# 4. GENERATE A RANDOM VALID ROUTE
# =====================================================

def generate_route(source, destination):
    def dfs(node, path, visited):
        if node == destination:
            return path

        neighbors = list(graph.get(node, {}).keys())
        random.shuffle(neighbors)

        for neighbor in neighbors:
            if neighbor not in visited:
                result = dfs(
                    neighbor,
                    path + [neighbor],
                    visited | {neighbor}
                )

                if result:
                    return result

        return None

    return dfs(source, [source], {source})


# =====================================================
# 5. INITIALIZE POPULATION
# =====================================================

def initialize_population(source, destination):
    population = []

    for _ in range(POPULATION_SIZE * 30):
        if len(population) >= POPULATION_SIZE:
            break

        route = generate_route(source, destination)

        if route and valid_route(route, source, destination):
            population.append(route)

    if not population:
        raise ValueError(
            "No valid route exists between these locations."
        )

    # Duplicates are allowed if the network has few possible routes
    while len(population) < POPULATION_SIZE:
        population.append(random.choice(population).copy())

    return population


# =====================================================
# 6. SELECTION: TOURNAMENT SELECTION
# =====================================================

def selection(population):
    candidates = random.sample(
        population,
        min(TOURNAMENT_SIZE, len(population))
    )

    return min(candidates, key=route_time)


# =====================================================
# 7. CROSSOVER
# Combine parents at a common intermediate intersection
# =====================================================

def crossover(parent1, parent2, source, destination):
    common_nodes = [
        node for node in parent1[1:-1]
        if node in parent2[1:-1]
    ]

    if not common_nodes:
        return parent1.copy(), parent2.copy()

    point = random.choice(common_nodes)

    i = parent1.index(point)
    j = parent2.index(point)

    child1 = parent1[:i] + parent2[j:]
    child2 = parent2[:j] + parent1[i:]

    if not valid_route(child1, source, destination):
        child1 = parent1.copy()

    if not valid_route(child2, source, destination):
        child2 = parent2.copy()

    return child1, child2


# =====================================================
# 8. MUTATION
# Replace a route with a newly generated valid route
# =====================================================

def mutation(route, source, destination):
    if random.random() < MUTATION_RATE:
        new_route = generate_route(source, destination)

        if new_route and valid_route(
            new_route, source, destination
        ):
            return new_route

    return route.copy()


# =====================================================
# 9. MAIN GENETIC ALGORITHM
# =====================================================

def genetic_algorithm(source, destination):
    if source not in graph or destination not in graph:
        raise ValueError("Source or destination is not in the graph.")

    if source == destination:
        return [source]

    population = initialize_population(source, destination)

    best_route = min(population, key=route_time)

    for generation in range(GENERATIONS):
        population.sort(key=route_time)

        # Elitism: preserve the best routes
        new_population = [
            route.copy()
            for route in population[:ELITE_COUNT]
        ]

        while len(new_population) < POPULATION_SIZE:
            parent1 = selection(population)
            parent2 = selection(population)

            child1, child2 = crossover(
                parent1, parent2, source, destination
            )

            child1 = mutation(child1, source, destination)
            child2 = mutation(child2, source, destination)

            new_population.append(child1)

            if len(new_population) < POPULATION_SIZE:
                new_population.append(child2)

        population = new_population

        current_best = min(population, key=route_time)

        if route_time(current_best) < route_time(best_route):
            best_route = current_best.copy()

    return best_route


# =====================================================
# 10. DISPLAY RESULTS
# =====================================================

if __name__ == "__main__":
    random.seed()

    print("Available locations:", ", ".join(sorted(graph)))

    source = input("Enter source location: ").strip().upper()
    destination = input("Enter destination location: ").strip().upper()

    try:
        best_route = genetic_algorithm(source, destination)

        distance = route_distance(best_route)
        time_minutes = route_time(best_route)

        print("\n========== DRIVING ROUTE ==========")
        print("Source:", source)
        print("Destination:", destination)
        print("Best route:", " -> ".join(best_route))
        print(f"Total distance: {distance:.2f} km")
        print(
            f"Estimated journey time: "
            f"{int(time_minutes // 60)} hours "
            f"{int(time_minutes % 60)} minutes"
        )
        print(f"Total time in minutes: {time_minutes:.2f}")

        print("\nRoad-by-road details:")
        for i in range(len(best_route) - 1):
            u = best_route[i]
            v = best_route[i + 1]
            road = graph[u][v]
            minutes = road["distance"] / road["speed"] * 60

            print(
                f"{u} -> {v}: "
                f"{road['distance']:.2f} km, "
                f"{road['speed']} km/h, "
                f"{minutes:.2f} minutes"
            )

    except ValueError as error:
        print("Error:", error)


Available locations: A, B, C, D, E, F, G, H, I, J
Enter source location: D
Enter destination location: H

========== DRIVING ROUTE ==========
Source: D
Destination: H
Best route: D -> F -> H
Total distance: 6.00 km
Estimated journey time: 0 hours 9 minutes
Total time in minutes: 9.00

Road-by-road details:
D -> F: 3.00 km, 40 km/h, 4.50 minutes
F -> H: 3.00 km, 40 km/h, 4.50 minutes
